# Baseline Modelling — Polymarket → Gold 5-min Returns

Pipeline summary (this notebook covers steps 1-9 of the design brief):

1. **Load** the latest `gold_panel_<date>.csv` produced by the feature-engineering notebook.
2. **Load** the Bloomberg target (`GOLD USD SPOT PER OZ`, `Close`).
3. Build the target `y`: **log-return over the past 5 min, lagged −1** so that predictors at `t` predict the return realised between `t` and `t+1` (≈ +5 min).
4. Attach **auto-regressive (AR) features** to the predictors.
5. Build `X`, `y` (scikit-learn ready). Build `X_traditional` from the remaining Bloomberg indicators (loaded but not used for fitting yet).
6. **Flexible walk-forward modelling framework** with three models (Linear, LSTM, Random Forest) × four window schemes (fixed 120 / 240 / 300 + expanding). LSTM uses **batched test blocks (12 obs per retrain)** to stay tractable.
7. **Walk-forward evaluation** (train on window → predict next observation, or next 12 for LSTM).
8. **Logging**: every run appends a row to `Results/runs_log.txt` plus a detailed `.json` side-car (model spec, metrics, data hash, run timestamp).
9. **Persistence**: models are saved as `Models/<MODEL>_<WINDOW>_<DATA-DATE>.pkl|.keras`. If a matching artefact already exists and the input CSV hasn't changed, training is skipped and the model is reloaded.

> **Open questions / critical issues are collected in `NOTES_modelling_baseline.md` — please read it before interpreting results.**


## 0. Config

In [ ]:
# %% ── CELL 0 : CONFIG ───────────────────────────────────────────────────────
from pathlib import Path

# Folders
DATA_DIR     = Path("./Data")
MODELS_DIR   = Path("./Models")
RESULTS_DIR  = Path("./Results")
for p in (MODELS_DIR, RESULTS_DIR):
    p.mkdir(exist_ok=True, parents=True)

# File patterns
GOLD_PANEL_GLOB = "gold_panel_*.csv"          # dynamic date in file name
BLOOMBERG_XLSX  = DATA_DIR / "Indicators-Data-bloomberg.xlsx"
TARGET_SHEET    = "GOLD USD SPOT PER OZ"
TARGET_COL      = "Close"

# Modelling
HORIZON_STEPS   = 1            # how many 5-min bars ahead we predict (t+1)
BAR_MINUTES     = 5            # native cadence of the Bloomberg close series
AR_LAGS         = [1, 2, 3, 6, 12]           # past-return lags to include (5, 10, 15, 30, 60 min)
AR_MA_WINDOWS   = [3, 6, 12, 36]             # rolling-mean windows on past returns
WINDOW_SCHEMES  = {                          # {label: ("fixed"|"expanding", size)}
    "fixed120":  ("fixed", 120),
    "fixed240":  ("fixed", 240),
    "fixed300":  ("fixed", 300),
    "expanding": ("expanding", 120),         # 120 = minimum training size before first prediction
}
LSTM_TEST_BLOCK = 12           # LSTM retrain cadence — predict next 12 obs per refit
LSTM_SEQ_LEN    = 12           # look-back length fed into the LSTM (≈ 1 hour)
LSTM_EPOCHS     = 8
LSTM_BATCH      = 32
RF_N_ESTIMATORS = 200
RANDOM_STATE    = 42
MAX_FEATURES_PREFILTER = 150   # crude variance prefilter to keep linear/LSTM tractable
                               # (full feature selection happens in the *next* thread)

# Evaluation toggles
FORCE_RETRAIN   = False        # set True to ignore cached models
DRY_RUN_ROWS    = None         # e.g. 1000 to debug on a slice; None = full


## 1. Dependencies

In [ ]:
# %% ── CELL 1 : DEPENDENCIES ─────────────────────────────────────────────────
import os, re, json, hashlib, joblib, warnings, datetime as dt
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge
from sklearn.ensemble    import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics     import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold

# TensorFlow / Keras (LSTM) — imported lazily so the notebook still runs if TF isn't installed
_TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(RANDOM_STATE)
except Exception as _tf_err:
    _TF_AVAILABLE = False
    print("⚠️  TensorFlow not importable — LSTM runs will be skipped.", _tf_err)

np.random.seed(RANDOM_STATE)
print("✅ Dependencies loaded. TF:", _TF_AVAILABLE)


## 2. Locate the latest gold panel CSV

The feature-engineering pipeline writes `gold_panel_<YYYY-MM-DD>.csv` to `./Data`. Here we pick the file with the most recent date in its name (NOT file mtime — mtime can be misleading if the file was merely copied).

In [ ]:
# %% ── CELL 2 : LOAD LATEST GOLD PANEL ───────────────────────────────────────
_date_rx = re.compile(r"gold_panel_(\d{4}-\d{2}-\d{2})\.csv$")

def find_latest_panel(folder: Path) -> tuple[Path, str]:
    candidates = []
    for p in folder.glob(GOLD_PANEL_GLOB):
        m = _date_rx.search(p.name)
        if m:
            candidates.append((m.group(1), p))
    if not candidates:
        raise FileNotFoundError(f"No files matching {GOLD_PANEL_GLOB} in {folder}")
    candidates.sort(key=lambda t: t[0])   # lexicographic sort works for ISO dates
    return candidates[-1][1], candidates[-1][0]

panel_path, panel_date = find_latest_panel(DATA_DIR)
print(f"Latest panel: {panel_path.name}  (data date = {panel_date})")

X_raw = pd.read_csv(panel_path, parse_dates=["scraped_at"]).set_index("scraped_at").sort_index()
if DRY_RUN_ROWS:
    X_raw = X_raw.iloc[-DRY_RUN_ROWS:]
print(f"X_raw shape : {X_raw.shape}   range: {X_raw.index.min()} → {X_raw.index.max()}")


## 3. Load Bloomberg indicators — build target and X_traditional

The Bloomberg workbook has one sheet per indicator, each with 5 columns (`Date, Open, High, Low, Close`). We keep the **Close** column of every sheet. `GOLD USD SPOT PER OZ` becomes the target; the rest go into `X_traditional`.

In [ ]:
# %% ── CELL 3 : LOAD BLOOMBERG SHEETS ────────────────────────────────────────
def load_bloomberg(xlsx_path: Path) -> dict[str, pd.DataFrame]:
    '''Load every Bloomberg sheet that has a `Date` column.
       Some sheets use `Close`, some use `Last Price` — we normalise both to `close`.'''
    xl = pd.ExcelFile(xlsx_path)
    out = {}
    for sheet in xl.sheet_names:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
        if df.empty or "Date" not in df.columns:
            continue
        df = df.rename(columns={c: c.strip() for c in df.columns})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        # normalise the price column name
        if "Close" in df.columns:
            df = df.rename(columns={"Close": "close"})
        elif "Last Price" in df.columns:
            df = df.rename(columns={"Last Price": "close"})
        else:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            if not numeric_cols:
                continue
            df = df.rename(columns={numeric_cols[-1]: "close"})
        out[sheet] = df
    return out

bb = load_bloomberg(BLOOMBERG_XLSX)
print("Bloomberg sheets loaded:", list(bb.keys()))
gold = bb[TARGET_SHEET][["close"]].rename(columns={"close": "gold_close"})
print("Gold close range:", gold.index.min(), "→", gold.index.max(), "| rows:", len(gold))


### 3.1 Build the target `y`

We want a **regression** target on **future returns**, not prices.

\[
y_t = \log\!\left(\frac{P_{t+1}}{P_t}\right)
\]

so that predictors available at time `t` forecast the realised 5-min log-return over `[t, t+1]`. Equivalently: take the 5-min log-return series and shift it by `-1`.

**Why log-returns vs simple returns?** Additivity across time (log-returns sum), symmetry, and far more stable numerics for small moves — standard in high-frequency finance.

In [ ]:
# %% ── CELL 3.1 : BUILD y (log-returns over 5 min, lagged -1) ───────────────
# Bloomberg sheets list rows newest-first; sort_index() already fixed that.
gold["logret_5m"] = np.log(gold["gold_close"]).diff()       # return realised between t-1 and t
gold["y"]         = gold["logret_5m"].shift(-HORIZON_STEPS) # shift forward = realised over [t, t+1]
# (gold.index = t. gold["y"].loc[t] = log(P_{t+1}) - log(P_t), i.e. the next-bar return.)
print(gold[["gold_close","logret_5m","y"]].tail(6))


### 3.2 Auto-regressive (AR) features

**Design choice — rationale:**
Gold returns at 5-min cadence exhibit small but exploitable **short-term momentum / mean-reversion** and strong **volatility clustering**. Any non-trivial baseline must give the model access to its own past, otherwise we're asking Polymarket features alone to beat a signal that's already in the price path.

I'm adding three groups of AR features, all computed on `logret_5m` (which is *already observed* at time `t`, so no look-ahead):

1. **Lagged returns** at 1, 2, 3, 6, 12 bars (5, 10, 15, 30, 60 min). Captures raw momentum.
2. **Rolling means of past returns** over 3, 6, 12, 36 bars. Same idea as the lectures' moving-average baseline; a denoised direction estimate.
3. **Rolling std (realised vol proxy)** over 12 and 36 bars. Lets tree models condition on the current vol regime — classic HAR-RV intuition.

All features use only information up to and including `t`, so there is **no leakage**. They are attached to the `X` predictor matrix after timestamp alignment.

In [ ]:
# %% ── CELL 3.2 : AR FEATURES ────────────────────────────────────────────────
def build_ar_features(logret: pd.Series,
                      lags=AR_LAGS,
                      ma_windows=AR_MA_WINDOWS,
                      vol_windows=(12, 36)) -> pd.DataFrame:
    feats = {}
    for L in lags:
        feats[f"ar_ret_lag{L}"] = logret.shift(L)
    for W in ma_windows:
        feats[f"ar_ret_ma{W}"]  = logret.shift(1).rolling(W).mean()
    for W in vol_windows:
        feats[f"ar_ret_std{W}"] = logret.shift(1).rolling(W).std()
    return pd.DataFrame(feats)

ar_feats = build_ar_features(gold["logret_5m"])
print("AR feature matrix shape:", ar_feats.shape)
print(ar_feats.tail(3))


## 4. Align Polymarket features to Bloomberg's 5-min grid, assemble `X` and `y`

Polymarket is scraped every ~5 min but the timestamps are **not on the clock** (14:15:52, 14:21:07, …), whereas Bloomberg prices sit exactly on `:00/:05/:10/…`. To merge them I use `merge_asof` with a 5-min tolerance and `direction="backward"`: for every Bloomberg bar, I grab the **latest** Polymarket snapshot strictly at or before that bar. This is the only look-ahead-safe way to align the two clocks.

After alignment we drop rows with no target (the last bar and any pre-warmup AR rows).

In [ ]:
# %% ── CELL 4 : ALIGN + BUILD X, y ───────────────────────────────────────────
# 1) Restrict Bloomberg to the period covered by the panel
start, end = X_raw.index.min().floor("5min"), X_raw.index.max().ceil("5min")
gold_aligned = gold.loc[(gold.index >= start) & (gold.index <= end)].copy()
print(f"Bloomberg bars in overlap window: {len(gold_aligned)}")

# 2) merge_asof: for each 5-min Bloomberg bar, pull the latest polymarket snapshot ≤ bar
X_raw_sorted = X_raw.sort_index()
merged = pd.merge_asof(
    left      = gold_aligned.reset_index().rename(columns={"Date":"ts"}),
    right     = X_raw_sorted.reset_index().rename(columns={"scraped_at":"ts"}),
    on        = "ts",
    direction = "backward",
    tolerance = pd.Timedelta(f"{BAR_MINUTES}min"),
).set_index("ts")

# 3) Pull in AR features (index already Bloomberg bars)
merged = merged.join(ar_feats, how="left")

# 4) Separate into X and y
y = merged["y"]
polymarket_cols = [c for c in X_raw.columns]
ar_cols         = list(ar_feats.columns)
feature_cols    = polymarket_cols + ar_cols
X = merged[feature_cols].copy()

# 5) Drop rows where y or AR features are undefined
valid = y.notna() & merged[ar_cols].notna().all(axis=1)
X, y = X.loc[valid], y.loc[valid]

# 6) Column-level NA handling: forward-fill within polymarket cols (stale quote is the truth),
#    then fill remaining with 0 (market that didn't exist yet).
X[polymarket_cols] = X[polymarket_cols].ffill().fillna(0.0)

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"y sample stats: mean={y.mean():.2e}  std={y.std():.2e}  min={y.min():.2e}  max={y.max():.2e}")


### 4.1 Optional variance pre-filter (keeps LSTM/Ridge tractable)

We have ~2k columns and only ~5k rows. Linear models with 2k predictors are still fine (Ridge solves that easily), but a LSTM with 2k inputs is wasteful since ~95% of columns have near-zero variance at 5-min cadence. We drop near-constants, then keep the top-`MAX_FEATURES_PREFILTER` by variance. **This is not feature selection** — the proper wrapper/filter/embedded selection happens in the next thread on the merged dataset.

In [ ]:
# %% ── CELL 4.1 : LIGHT VARIANCE PREFILTER ───────────────────────────────────
vt = VarianceThreshold(threshold=1e-8)
vt.fit(X.values)
kept_mask  = vt.get_support()
kept_cols  = X.columns[kept_mask]
print(f"After VarianceThreshold: {len(kept_cols)} / {X.shape[1]} columns survive.")

# Keep the top-N by variance (heuristic only — cheap to change later)
variances    = X[kept_cols].var().sort_values(ascending=False)
top_cols     = variances.head(MAX_FEATURES_PREFILTER).index.tolist()
# Always keep AR features regardless of variance ranking
for c in ar_cols:
    if c in X.columns and c not in top_cols:
        top_cols.append(c)

X_all = X.copy()                 # untouched full matrix (for reference)
X     = X[top_cols]
print(f"X shape after prefilter : {X.shape}")


## 5. Build `X_traditional` (NOT used in modelling yet — stored for later)

In [ ]:
# %% ── CELL 5 : X_traditional ───────────────────────────────────────────────
traditional_sheets = [s for s in bb if s != TARGET_SHEET]
frames = []
for s in traditional_sheets:
    sdf = bb[s][["close"]].rename(columns={"close": f"{s}__close"})
    frames.append(sdf)
X_traditional_raw = pd.concat(frames, axis=1).sort_index()

# Align to the same 5-min Bloomberg grid as X
X_traditional = X_traditional_raw.reindex(X.index, method="nearest", tolerance=pd.Timedelta(f"{BAR_MINUTES}min"))
# Convert to log-returns (stationary) so they match y's world
X_traditional = np.log(X_traditional).diff().rename(columns=lambda c: c.replace("__close","__logret"))
X_traditional = X_traditional.loc[X.index].ffill().fillna(0.0)
print("X_traditional shape:", X_traditional.shape)
print(X_traditional.head(3))


## 6. Modelling framework (walk-forward)

Everything below is organised so that **adding / removing a model is a one-line change**:

```python
MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}
```

A *model factory* returns an object exposing `.fit(X,y)` and `.predict(X)`. The walk-forward loop is identical for all of them; only LSTM uses the **batched retraining** path (one refit every `LSTM_TEST_BLOCK` obs).

### Cross-validation strategy
For each `(model, window_scheme)` combo we run an **expanding or fixed-size walk-forward** where at each step the model is trained on `[t-window, t-1]` (fixed) or `[start, t-1]` (expanding) and predicts `y_t`. This is the closest scikit-learn analogue to your "train on rolling window, test on training_size+1" spec. For LSTM we predict 12 steps in one go before refitting — same walk-forward, just coarser grid.

In [ ]:
# %% ── CELL 6 : HELPERS — WINDOW INDICES & METRICS ──────────────────────────
def walk_forward_indices(n: int, kind: str, size: int, step: int = 1):
    '''Yield (train_idx_slice, test_idx_slice) tuples over 0..n-1.
       kind='fixed'      -> train = [i-size : i]
       kind='expanding'  -> train = [0      : i]
       test is [i : i+step].'''
    assert kind in ("fixed","expanding")
    start = size                         # first index for which we have enough history
    for i in range(start, n, step):
        if kind == "fixed":
            tr = slice(i-size, i)
        else:
            tr = slice(0, i)
        te = slice(i, min(i+step, n))
        if te.stop <= te.start:
            break
        yield tr, te

def compute_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    m = {
        "n"       : int(len(y_true)),
        "rmse"    : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae"     : float(mean_absolute_error(y_true, y_pred)),
        "r2"      : float(r2_score(y_true, y_pred)) if len(y_true) > 1 else float("nan"),
        "dir_acc" : float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        "hit_rate_nonzero" : float(np.mean((y_true != 0) & (np.sign(y_true) == np.sign(y_pred)))
                                   / max((y_true != 0).mean(), 1e-9)),
    }
    return m


In [ ]:
# %% ── CELL 6.1 : MODEL FACTORIES ────────────────────────────────────────────
class ScaledRegressor:
    '''Linear / LSTM wrapper that standardises X (and optionally y) internally.
       Keeps the main loop model-agnostic.'''
    def __init__(self, core, scale_y=False):
        self.core, self.scale_y = core, scale_y
        self.xs = StandardScaler()
        self.ys = StandardScaler() if scale_y else None
    def fit(self, X, y):
        Xs = self.xs.fit_transform(X)
        if self.ys is not None:
            ys = self.ys.fit_transform(np.asarray(y).reshape(-1,1)).ravel()
            self.core.fit(Xs, ys)
        else:
            self.core.fit(Xs, y)
        return self
    def predict(self, X):
        Xs = self.xs.transform(X)
        pred = self.core.predict(Xs)
        if self.ys is not None:
            pred = self.ys.inverse_transform(np.asarray(pred).reshape(-1,1)).ravel()
        return np.asarray(pred).ravel()


def make_linear():
    # Ridge = safe default with 150+ columns and ~5k rows. Very cheap to retrain at every step.
    return ScaledRegressor(Ridge(alpha=1.0, random_state=RANDOM_STATE))

def make_rf():
    # Trees are scale-invariant — no wrapper needed; sklearn RF pickles fine.
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS, max_depth=None,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE,
    )

def _build_lstm(n_features: int, seq_len: int = LSTM_SEQ_LEN):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

class LSTMRegressor:
    '''LSTM with internal X/y scaling and a sliding-window transformer.
       predict(X) expects a 2-D array where row i corresponds to the "current" timestep;
       the model looks at the previous seq_len rows internally.'''
    def __init__(self, seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.seq_len, self.epochs, self.batch = seq_len, epochs, batch
        self.xs, self.ys = StandardScaler(), StandardScaler()
        self.model = None
        self._last_train_X = None   # used to build sequences that span the train→test boundary
    def _seq(self, Xs, ys=None):
        # For each i >= seq_len, build a window Xs[i-seq_len:i] → target ys[i]
        X_out, y_out = [], []
        for i in range(self.seq_len, len(Xs)):
            X_out.append(Xs[i-self.seq_len:i])
            if ys is not None:
                y_out.append(ys[i])
        X_out = np.asarray(X_out)
        return (X_out, np.asarray(y_out)) if ys is not None else X_out
    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y).reshape(-1,1)
        Xs = self.xs.fit_transform(X)
        ys = self.ys.fit_transform(y).ravel()
        Xseq, yseq = self._seq(Xs, ys)
        self.model = _build_lstm(X.shape[1], self.seq_len)
        es = EarlyStopping(patience=3, restore_best_weights=True, monitor="loss")
        self.model.fit(Xseq, yseq, epochs=self.epochs, batch_size=self.batch,
                       verbose=0, callbacks=[es])
        self._last_train_X = Xs[-self.seq_len:]   # keep tail for boundary-spanning sequences
        return self
    def predict(self, X):
        X = np.asarray(X)
        Xs = self.xs.transform(X)
        # Prepend the tail of train so the first test rows have their look-back window.
        Xs_ext = np.vstack([self._last_train_X, Xs]) if self._last_train_X is not None else Xs
        preds = []
        for i in range(self.seq_len, len(Xs_ext)):
            window = Xs_ext[i-self.seq_len:i][None, ...]
            preds.append(self.model.predict(window, verbose=0)[0,0])
        preds = np.asarray(preds)
        # inverse-scale back to raw return units
        return self.ys.inverse_transform(preds.reshape(-1,1)).ravel()

def make_lstm():
    if not _TF_AVAILABLE:
        return None
    return LSTMRegressor()

MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}


### 6.2 Walk-forward runner

For `linear` and `rf` we use `step=1` (one-step-ahead, the finest grid possible). For `lstm` we use `step=LSTM_TEST_BLOCK` (=12, ~1 hour per refit) per your spec 6.1. The same helper covers both.

In [ ]:
# %% ── CELL 6.2 : WALK-FORWARD RUNNER ────────────────────────────────────────
def walk_forward(model_name: str, scheme_name: str, X: pd.DataFrame, y: pd.Series):
    kind, size = WINDOW_SCHEMES[scheme_name]
    step = LSTM_TEST_BLOCK if model_name == "lstm" else 1
    # expanding: we still need a minimum history (size) before the first prediction
    y_true, y_pred, ts_pred = [], [], []
    n = len(X)
    iters = list(walk_forward_indices(n, kind, size, step=step))
    for k, (tr, te) in enumerate(iters):
        factory = MODEL_REGISTRY[model_name]
        mdl = factory()
        if mdl is None:
            return None
        mdl.fit(X.iloc[tr].values, y.iloc[tr].values)
        p  = mdl.predict(X.iloc[te].values)
        y_true.extend(y.iloc[te].values.tolist())
        y_pred.extend(np.asarray(p).tolist())
        ts_pred.extend(y.iloc[te].index.tolist())
        if (k % 50 == 0):
            print(f"  [{model_name}/{scheme_name}] step {k+1}/{len(iters)} (train={tr.stop-tr.start}, test={te.stop-te.start})")
    return {
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "timestamps": ts_pred,
        "final_model": mdl,
    }


## 7. Logging and model persistence

- `Results/runs_log.txt` — one human-readable row per `(model, window, data-date)` run.
- `Results/runs_log.jsonl` — the same info as JSON for reproducibility / downstream comparison across weeks.
- `Models/<MODEL>_<WINDOW>_<DATA-DATE>.(pkl|keras)` — the **final-step** fitted model (trained on the last available window), so you can reload it without rerunning the whole walk-forward.

**"Retrain only if data changed"** logic:
- We compute `sha1` of the input gold panel CSV (content, not filename) and store it inside the sidecar JSON. On the next run, if a cached model for that `(model, window, data-hash)` triple exists and `FORCE_RETRAIN is False`, we just reload.

In [ ]:
# %% ── CELL 7 : PERSISTENCE HELPERS ─────────────────────────────────────────
def file_sha1(path: Path, bufsize=1<<20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bufsize), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_HASH = file_sha1(panel_path)
print("Data SHA1:", DATA_HASH[:12], "…")

def artefact_paths(model_name: str, scheme: str, data_date: str) -> tuple[Path, Path]:
    stem = f"{model_name.upper()}_{scheme}_{data_date}"
    ext = ".keras" if model_name == "lstm" else ".pkl"
    return MODELS_DIR / f"{stem}{ext}", RESULTS_DIR / f"{stem}.json"

def cached_run_valid(meta_path: Path, data_hash: str) -> bool:
    if not meta_path.exists():
        return False
    try:
        meta = json.loads(meta_path.read_text())
        return meta.get("data_sha1") == data_hash and not FORCE_RETRAIN
    except Exception:
        return False

def save_artefacts(model_name, scheme, data_date, data_hash, metrics, result, X_cols):
    mdl_path, meta_path = artefact_paths(model_name, scheme, data_date)
    mdl = result["final_model"]
    # --- save final-step fitted model ---
    if model_name == "lstm":
        # LSTMRegressor wraps a keras model; save the keras weights + scalers separately
        mdl.model.save(mdl_path)
        joblib.dump({"xs": mdl.xs, "ys": mdl.ys, "seq_len": mdl.seq_len,
                     "last_train_X": mdl._last_train_X},
                    mdl_path.with_suffix(".scalers.pkl"))
    else:
        joblib.dump(mdl, mdl_path)
    # --- side-car metadata ---
    meta = {
        "model"       : model_name,
        "window"      : scheme,
        "data_date"   : data_date,
        "data_sha1"   : data_hash,
        "run_utc"     : dt.datetime.utcnow().isoformat(timespec="seconds"),
        "n_features"  : len(X_cols),
        "feature_sample": X_cols[:15],
        "metrics"     : metrics,
        "config"      : {
            "horizon_steps" : HORIZON_STEPS,
            "ar_lags"       : AR_LAGS,
            "ar_ma_windows" : AR_MA_WINDOWS,
            "lstm_seq_len"  : LSTM_SEQ_LEN,
            "lstm_block"    : LSTM_TEST_BLOCK,
            "rf_estimators" : RF_N_ESTIMATORS,
            "prefilter_topN": MAX_FEATURES_PREFILTER,
        },
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str))
    # --- append to master log ---
    with open(RESULTS_DIR / "runs_log.txt", "a") as f:
        f.write(
            f"{meta['run_utc']}  {model_name:7s}  {scheme:10s}  "
            f"data={data_date}  rmse={metrics['rmse']:.5e}  "
            f"mae={metrics['mae']:.5e}  r2={metrics['r2']:+.4f}  "
            f"dir_acc={metrics['dir_acc']:.3f}\n")
    with open(RESULTS_DIR / "runs_log.jsonl", "a") as f:
        f.write(json.dumps(meta, default=str) + "\n")
    return meta_path, mdl_path


## 8. Run the full grid

`MODELS × WINDOW_SCHEMES`. Cached artefacts are reused if the data hasn't changed.

In [ ]:
# %% ── CELL 8 : RUN THE GRID ─────────────────────────────────────────────────
MODELS_TO_RUN = ["linear", "rf", "lstm"]   # edit this list freely; registry is extensible
all_results = []

for model_name in MODELS_TO_RUN:
    if model_name == "lstm" and not _TF_AVAILABLE:
        print("⏭  Skipping LSTM (TensorFlow not available).")
        continue
    for scheme in WINDOW_SCHEMES:
        mdl_path, meta_path = artefact_paths(model_name, scheme, panel_date)
        if cached_run_valid(meta_path, DATA_HASH):
            meta = json.loads(meta_path.read_text())
            print(f"✓ Cached: {model_name}/{scheme} — rmse={meta['metrics']['rmse']:.3e}  dir_acc={meta['metrics']['dir_acc']:.3f}")
            all_results.append(meta)
            continue
        print(f"▶ Training {model_name}/{scheme} …")
        res = walk_forward(model_name, scheme, X, y)
        if res is None:
            continue
        metrics = compute_metrics(res["y_true"], res["y_pred"])
        save_artefacts(model_name, scheme, panel_date, DATA_HASH, metrics, res, list(X.columns))
        all_results.append({"model": model_name, "window": scheme, "data_date": panel_date, "metrics": metrics})
        print(f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}")


## 9. Summary table

In [ ]:
# %% ── CELL 9 : SUMMARY ─────────────────────────────────────────────────────
rows = []
for r in all_results:
    m = r.get("metrics", r)
    rows.append({
        "model"   : r.get("model"),
        "window"  : r.get("window"),
        "date"    : r.get("data_date"),
        "rmse"    : m.get("rmse"),
        "mae"     : m.get("mae"),
        "r2"      : m.get("r2"),
        "dir_acc" : m.get("dir_acc"),
        "n"       : m.get("n"),
    })
summary = pd.DataFrame(rows).sort_values(["model","window"])
print(summary.to_string(index=False))
# Also persist a wide-format snapshot for easy run-to-run comparison
summary.to_csv(RESULTS_DIR / f"summary_{panel_date}.csv", index=False)


## 10. (OPTIONAL — commented out) Handling the 23:00 daily gap

Bloomberg's gold feed has a ~1h daily break starting around 23:00. The polymarket markets keep moving during that window, and the FIRST bar after re-open probably reflects all the information accumulated during the break. Several ways to model this — none is obviously right, so I'm **leaving them commented out** pending your evaluation:

1. **Mask the gap**: drop the first post-break bar from training (it's a return over ~65 min, not 5 min — it breaks the "homogeneous sampling" assumption of all our models).
2. **Tag it with a dummy**: add an `is_post_break` feature. Lets RF/Linear learn a separate intercept for that bar without us removing data.
3. **Accumulate polymarket movement during the gap** as an extra feature (e.g., `poly_return_during_break`), so the model sees what happened during the blackout.
4. **Session-aware scaling**: group returns by session and standardise within each to avoid inflated vol on the reopen bar.

In [ ]:
# %% ── CELL 10 : BREAK-HANDLING (COMMENTED OUT) ─────────────────────────────
# Uncomment ONE of the blocks below after deciding which approach to adopt.

# # -------- Approach 1: drop the first bar after each break > 1 hour --------
# gap_mask = X.index.to_series().diff() > pd.Timedelta("60min")
# X = X.loc[~gap_mask]
# y = y.loc[X.index]

# # -------- Approach 2: add an is_post_break dummy feature ------------------
# X["is_post_break"] = (X.index.to_series().diff() > pd.Timedelta("60min")).astype(int).values

# # -------- Approach 3: polymarket change during the break ------------------
# # For each post-break bar, add sum of abs changes in bestAsk columns during the gap.
# # (Implementation requires keeping the raw high-frequency X_raw around.)
# # poly_diff = X_raw.diff().abs().sum(axis=1)
# # gap_feature = poly_diff.resample("1min").sum().reindex(X.index, method="nearest").fillna(0)
# # X["poly_movement_during_break"] = gap_feature

# # -------- Approach 4: per-session standardisation of y --------------------
# # sessions = (X.index.to_series().diff() > pd.Timedelta("60min")).cumsum()
# # y = y.groupby(sessions).transform(lambda s: (s - s.mean()) / (s.std() + 1e-12))


## 11. How to reload a model later (no retraining)

In [ ]:
# %% ── CELL 11 : MODEL RELOAD EXAMPLE ───────────────────────────────────────
# Example — load the expanding-window Linear model trained on today's data:
# mdl_path, meta_path = artefact_paths("linear", "expanding", panel_date)
# model = joblib.load(mdl_path)
# meta  = json.loads(meta_path.read_text())
# print("Reloaded:", meta["model"], meta["window"], "metrics:", meta["metrics"])
# # For LSTM:
# # from tensorflow.keras.models import load_model
# # keras_model = load_model(mdl_path)
# # scalers = joblib.load(mdl_path.with_suffix(".scalers.pkl"))
